# Speech Separation tiếng Việt: baseline → fine-tune → so sánh

Notebook chạy trên **Google Colab (GPU)**. Chạy lần lượt từ trên xuống.

| Bước | Việc | Thời gian ước tính |
|---|---|---|
| 1–3 | Chuẩn bị môi trường, lấy code (Drive hoặc GitHub), cài thư viện | ~3 phút |
| 4 | Tải VIVOS + trộn dữ liệu (lưu lên Drive để lần sau dùng lại) | ~10–20 phút |
| 5 | **Baseline**: đánh giá model tiếng Anh gốc `sepformer-whamr16k` | tùy số mẫu |
| 6 | **Fine-tune** trên dữ liệu tiếng Việt | lâu nhất — có lưu checkpoint lên Drive |
| 7 | Đánh giá model sau fine-tune | tùy số mẫu |
| 8 | Bảng so sánh + biểu đồ + nghe thử | |

Runtime → Change runtime type → **T4 GPU**.

**Trước khi chạy:** upload `vietnamese-voice-qa.zip` vào `My Drive/vvqa/` (tạo thư mục `vvqa` nếu chưa có).

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Kết nối Google Drive (lưu dữ liệu, checkpoint, kết quả để không mất khi Colab ngắt)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/vvqa'          # đổi nếu muốn
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints/separation/finetuned'
METRIC_DIR = f'{DRIVE_ROOT}/metrics'
DATA_TAR   = f'{DRIVE_ROOT}/data/separation_vivos.tar'
import os
for d in (CKPT_DIR, METRIC_DIR, os.path.dirname(DATA_TAR)):
    os.makedirs(d, exist_ok=True)

In [ ]:
# 3. Lấy code + cài thư viện
#   'drive' : giải nén file zip đã upload lên Drive (chưa cần push GitHub)
#   'github': clone nhánh feat/separation từ GitHub
CODE_SOURCE = 'drive'
CODE_ZIP    = f'{DRIVE_ROOT}/vietnamese-voice-qa.zip'
REPO_URL    = 'https://github.com/thieuphong220203/vietnamese-voice-qa.git'
BRANCH      = 'feat/separation'

%cd /content
!rm -rf vietnamese-voice-qa
if CODE_SOURCE == 'drive':
    assert os.path.exists(CODE_ZIP), f'Không thấy {CODE_ZIP} — upload file zip vào thư mục vvqa trên Drive'
    !unzip -q "{CODE_ZIP}" -d /content
else:
    !git clone -b {BRANCH} {REPO_URL}
%cd /content/vietnamese-voice-qa

# Colab đã có torch/torchaudio/transformers; chỉ cần thêm speechbrain + thư viện đọc dữ liệu
!pip install -q "speechbrain>=1.0" huggingface_hub pyyaml

## 4. Dữ liệu: VIVOS → trộn 2 người nói

- `train`: người nói VIVOS/train (trừ vài người giữ làm `valid`), cắt đoạn 4 giây
- `valid`: người nói giữ lại, dùng chọn checkpoint tốt nhất
- `test`: người nói VIVOS/test — **không bao giờ dùng để train**, giữ cố định để so sánh trước/sau

Lần đầu: tải VIVOS (khoảng 1.4 GB, từ Hugging Face; lỗi thì tự chuyển sang Zenodo) rồi trộn, sau đó nén lên Drive. Các lần sau chỉ giải nén từ Drive.

> Nếu cả hai nguồn đều lỗi: tải `vivos.tar.gz` thủ công, giải nén để có `data/raw/vivos/train/waves/...` và `data/raw/vivos/test/waves/...`, rồi chạy lại ô dưới **không có** `--download`.

In [ ]:
N_TRAIN, N_VALID, N_TEST = 4000, 200, 300   # giảm nếu muốn chạy thử nhanh

if os.path.exists(DATA_TAR):
    print('Dùng dữ liệu đã trộn sẵn trên Drive')
    !mkdir -p data/processed && tar -xf {DATA_TAR} -C data/processed
else:
    !python -m src.data.prepare_separation --download --vivos-root data/raw/vivos \
        --output-dir data/processed/separation \
        --n-train {N_TRAIN} --n-valid {N_VALID} --n-test {N_TEST}
    !tar -cf {DATA_TAR} -C data/processed separation
!wc -l data/processed/separation/*.csv

In [ ]:
# Nghe thử 1 mẫu test: mix / s1 / s2
import pandas as pd
from IPython.display import Audio, display
test_df = pd.read_csv('data/processed/separation/test.csv')
row = test_df.iloc[0]
print('s1:', row.s1_text, '| s2:', row.s2_text, f'| s1 to hơn {row.gain_db} dB')
for key in ('mix', 's1', 's2'):
    print(key); display(Audio(f'data/processed/separation/{row[key]}'))

## 5. Baseline — model tiếng Anh gốc

`--asr` tính thêm WER của PhoWhisper trên câu của người nói chính (s1):
- `wer_mix`: đưa thẳng audio trộn vào ASR (không có separation)
- `wer_primary`: tách giọng → chọn giọng to nhất → ASR (đúng như pipeline)
- `wer_clean`: s1 sạch gốc (giới hạn tốt nhất có thể)

In [ ]:
EVAL_LIMIT = None   # đặt ví dụ 50 để chạy thử nhanh
limit_arg = f'--limit {EVAL_LIMIT}' if EVAL_LIMIT else ''
!python -m src.separation.evaluate --csv data/processed/separation/test.csv \
    --tag baseline --asr {limit_arg} --output {METRIC_DIR}/separation_baseline.csv

## 6. Fine-tune

- Checkpoint lưu thẳng lên Drive: `last.pt` (để chạy tiếp), `best.pt` (SI-SNRi valid cao nhất).
- **Colab bị ngắt?** Chạy lại các ô 1–4, rồi chạy ô này — nó tự chạy tiếp từ `last.pt`.
- **CUDA out of memory?** Giảm `SEGMENT` xuống 2 hoặc 1.5.

In [ ]:
EPOCHS, SEGMENT, LR = 5, 3.0, 5e-5

resume = f'--resume {CKPT_DIR}/last.pt' if os.path.exists(f'{CKPT_DIR}/last.pt') else ''
!python -m src.separation.train \
    --train-csv data/processed/separation/train.csv \
    --valid-csv data/processed/separation/valid.csv \
    --output-dir {CKPT_DIR} --epochs {EPOCHS} --segment-seconds {SEGMENT} --lr {LR} \
    --batch-size 1 --grad-accum 4 --fp16 {resume}

## 7. Đánh giá sau fine-tune (cùng tập test)

In [ ]:
!python -m src.separation.evaluate --csv data/processed/separation/test.csv \
    --checkpoint {CKPT_DIR}/best.pt --tag finetuned --asr {limit_arg} \
    --output {METRIC_DIR}/separation_finetuned.csv

## 8. So sánh

In [ ]:
import json
import matplotlib.pyplot as plt

summaries = {tag: json.load(open(f'{METRIC_DIR}/separation_{tag}.summary.json'))
             for tag in ('baseline', 'finetuned') if os.path.exists(f'{METRIC_DIR}/separation_{tag}.summary.json')}
cols = ['samples', 'si_snr_mix', 'si_snr', 'si_snri', 'primary_accuracy', 'wer_mix', 'wer_primary', 'wer_clean', 'rtf']
table = pd.DataFrame(summaries).T.reindex(columns=cols)
display(table.round(3))
table.to_csv(f'{METRIC_DIR}/separation_comparison.csv')

log = pd.read_csv(f'{CKPT_DIR}/train_log.csv')
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
log.plot(x='epoch', y='train_loss', ax=ax[0], marker='o', title='Train loss (-SI-SNR)', legend=False)
log.plot(x='epoch', y='valid_si_snri', ax=ax[1], marker='o', title='Valid SI-SNRi (dB)', legend=False)
plt.tight_layout(); plt.savefig(f'{METRIC_DIR}/separation_training.png', dpi=150)

In [ ]:
# Nghe thử: giọng tách được trước và sau fine-tune cho cùng một mẫu
sample_id = test_df.iloc[0].id
for tag in ('baseline', 'finetuned'):
    for i in (1, 2):
        path = f'{DRIVE_ROOT}/audio/separation_{tag}/{sample_id}_est{i}.wav'
        if os.path.exists(path):
            print(tag, 'est', i); display(Audio(path))